# Creating the Bulls Draft Data

---

#### Importing Raw Draft Data

Raw data found on https://www.basketball-reference.com/teams/CHI/draft.html

Used button on site labeled **"Share & Export"** to see table s raw csv file, and copy and pasted draft data from 2000-most recent data. Data was saved in *draft_data.csv* in the *data* folder.

In [39]:
import pandas as pd
from pandas.errors import EmptyDataError
from pathlib import Path

df = pd.read_csv('../data/draft_data.csv')

df.head()

,Year,Lg,Rd,Pk,Player,College,G,MP,PTS,TRB,...,3P%,FT%,MP.1,PTS.1,TRB.1,AST.1,WS,WS/48,BPM,VORP
0,2025,NBA,1,12,Noa Essengue,NaN,2.0,6.0,0.0,0.0,...,0.000,NaN,3.0,0.0,0.0,0.0,-0.1,-0.417,-23.7,0.0
1,2025,NBA,2,45,Rocco Zikarsky (↳LAL ↳MIN),NaN,5.0,36.0,14.0,14.0,...,NaN,1.000,7.2,2.8,2.8,0.4,0.2,0.237,2.4,0.0
2,2024,NBA,1,11,Matas Buzelis,NaN,157.0,3759.0,1940.0,726.0,...,0.353,0.795,23.9,12.4,4.6,1.5,4.4,0.057,-1.6,0.4
3,2023,NBA,2,59,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2022,NBA,1,18,Dalen Terry,Arizona,218.0,2433.0,767.0,360.0,...,0.314,0.638,11.2,3.5,1.7,1.2,3.1,0.061,-2.6,-0.3


This is the raw data from basketball-reference.

There are a couple of problems with the data:
- Includes draft picks that were traded to other teams
- Includes present day stats, we need players' rookie year stats
- Some years have no draft picks, need to disregard

#### Drop Null Picks

Some seasons resulted in zero draft picks, but these are represented by empty rows

In [40]:
print(f'Full dataset shape: {df.shape[0]} rows')

draft_df = df.dropna(subset=['Player']).reset_index(drop=True)
print(f'Dataset after removing rows with missing Player names: {draft_df.shape[0]} rows')

Full dataset shape: 55 rows
Dataset after removing rows with missing Player names: 54 rows


#### Getting Rid of Traded Players


In [41]:
print(f'Full dataset shape: {draft_df.shape[0]} rows')

draft_df = draft_df[~draft_df['Player'].str.contains('↳')].reset_index(drop=True)
print(f'Dataset after removing traded players: {draft_df.shape[0]} rows')

Full dataset shape: 54 rows
Dataset after removing traded players: 39 rows


#### Creating Rookie Year Data Folders

A place to put all of the rookie year stats is needed to store all of the data. A folder was made manually to store csvs, but each csv file was made programatically based on what players are available. 

Some of the players' stats have different columns based on how much playtime they had, so the stats couldn;t be grouped into the same csv file right away. 

In [ ]:
# define folder to save files in
output_dir = Path('../data/rookie_stats_data')
output_dir.mkdir(parents=True, exist_ok=True)

# gather player names
player_names = (
    draft_df['Player']
    .dropna()
    .astype(str)
    .str.strip()
    .drop_duplicates()
)

created_files = []
existing_files = []

# create a CSV file for each player
for player_name in player_names:
    # file name is in the structure {first name}_{last_name}.csv
    filename = f"{player_name.replace(' ', '_')}.csv"
    file_path = output_dir / filename

    if file_path.exists():
        existing_files.append(filename)
        continue

    file_path.write_text('')
    created_files.append(filename)

print(f"Created {len(created_files)} CSV files in {output_dir}")
print(f"Skipped {len(existing_files)} existing files")


Created 39 CSV files in ..\data\rookie_stats_data


#### Cleaning the Rookie Stats Files

Some of the pages for the players' rookie years were empty or missing certain statistics. Those columns needed to be added and set to be null so that the rookie stats could be merged correctly.

In [43]:
# define folder to read files from
data_dir = Path('../data/rookie_stats_data')
csv_files = sorted(data_dir.glob('*.csv'))

file_data = {}
all_columns = []

# read each CSV file and collect column information
for csv_file in csv_files:
    try:
        df = pd.read_csv(csv_file)
        file_data[csv_file] = df

        for column in df.columns:
            if column not in all_columns:
                all_columns.append(column)
    except EmptyDataError:
        file_data[csv_file] = None

# unify columns across all files
if not all_columns:
    print('No column information was found in the CSV files.')
else:
    updated_files = []
    empty_files = []

    # fill missing columns with NaN and save updated files
    for csv_file, df in file_data.items():
        if df is None:
            fixed_df = pd.DataFrame([{column: pd.NA for column in all_columns}])
            empty_files.append(csv_file.name)
            missing_columns = all_columns.copy()
        else:
            missing_columns = [column for column in all_columns if column not in df.columns]
            fixed_df = df.copy()

            for column in missing_columns:
                fixed_df[column] = pd.NA

            fixed_df = fixed_df[all_columns]

        fixed_df.to_csv(csv_file, index=False)

        if missing_columns:
            updated_files.append((csv_file.name, missing_columns))

    print(f'Checked {len(csv_files)} files.')
    print(f'Unified column set has {len(all_columns)} columns.')
    print()

    # report results
    if empty_files:
        print('Empty files filled with NaN values:')
        for file_name in empty_files:
            print(f'  {file_name}')
        print()

    if updated_files:
        print('Files updated with missing columns:')
        for file_name, missing_columns in updated_files:
            print(f'  {file_name}: {missing_columns}')
    else:
        print('All files already had the same columns.')


Checked 39 files.
Unified column set has 1 columns.

All files already had the same columns.


#### Getting Rookie Year Data

Stats are currently listed as most recent season, and this project requires the rookie year stats. This was not possible programatically, so all stats were pasted in manually.

In [45]:
draft_df.columns

Index(['Year', 'Lg', 'Rd', 'Pk', 'Player', 'College', 'G', 'MP', 'PTS', 'TRB',
       'AST', 'FG%', '3P%', 'FT%', 'MP.1', 'PTS.1', 'TRB.1', 'AST.1', 'WS',
       'WS/48', 'BPM', 'VORP'],
      dtype='object')